# MEL — Smoke test QLoRA 4-bit
Ce notebook exécute un vrai smoke test GPU de 500 conversations. Il échoue immédiatement si aucun GPU CUDA n'est disponible. Le résultat attendu est `TRAINED_UNBENCHMARKED`, jamais une promotion automatique.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'GPU CUDA requis : activez un runtime GPU (T4/L4/A100) puis relancez Run all.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!rm -rf /content/meliturgos-cloudflare
!git clone --branch candidate/mel-clean-autonomy --single-branch https://github.com/adrienlopezcarreras-pixel/meliturgos-cloudflare.git /content/meliturgos-cloudflare
%cd /content/meliturgos-cloudflare
!python -m pip install -U pip
!python -m pip install -r requirements-lora.txt ijson

## Pool Hugging Face compatible
Découvre automatiquement les adaptateurs PEFT/LoRA compatibles avec la base Cloudflare de MEL. La liste est auditée et enregistrée avant l’entraînement; les candidats seront benchmarkés séparément.

In [ ]:
!python -m pip install -U huggingface_hub
!mkdir -p artifacts/lora-data
!python scripts/discover-compatible-hf-loras.py --output artifacts/lora-data/hf-compatible-registry.json --limit-per-query 100
import json
from pathlib import Path
hfreg=json.loads(Path('artifacts/lora-data/hf-compatible-registry.json').read_text())
print('LoRA Hugging Face compatibles :', hfreg['compatible_count'])
print('Rejetés pour incompatibilité :', hfreg['rejected_count'])


In [ ]:
!mkdir -p artifacts/lora-data artifacts/lora-train
!python scripts/prepare-sharegpt-lora.py --variant both --max-conversations 500 --seed 42 --output artifacts/lora-data/sharegpt-mel-smoke-500.jsonl
!node scripts/create-lora-plan.mjs --dataset artifacts/lora-data/sharegpt-mel-smoke-500.jsonl --output artifacts/lora-data/lora-plan-smoke-500.json --epochs 1 --seed 42


In [ ]:
!python scripts/train-mel-lora.py --dataset artifacts/lora-data/sharegpt-mel-smoke-500.jsonl --plan artifacts/lora-data/lora-plan-smoke-500.json --output artifacts/lora-train/smoke-500 --epochs 1 --save-steps 50 --seed 42


In [ ]:
from pathlib import Path
import json
root = Path('artifacts/lora-train/smoke-500')
required = [root/'adapter_model.safetensors', root/'adapter_config.json', root/'training-evidence.json', root/'artifact-evidence.json', root/'lora-plan.json']
for p in required:
    assert p.is_file() and p.stat().st_size > 0, f'MISSING: {p}'
e = json.loads((root/'training-evidence.json').read_text())
a = json.loads((root/'artifact-evidence.json').read_text())
p = json.loads((root/'lora-plan.json').read_text())
assert e['status'] == 'TRAINED_UNBENCHMARKED'
assert e['training_mode'] == 'qlora-4bit-nf4'
assert e['quantization']['bits'] == 4
assert e['environment']['cuda_available'] is True
assert e['training_manifest_digest'] == p['training_manifest_digest'] == a['training_manifest_digest']
assert e['dataset_digest'] == p['dataset_digest'] == a['dataset_digest']
assert a['finetune_id'] is None
print(json.dumps({'status':'PASS','gpu':e['environment']['gpu'],'examples':e['dataset']['examples'],'train_loss':e['training_metrics']['train_loss'],'global_step':e['training_metrics']['global_step'],'plan_id':e['plan_id'],'manifest':e['training_manifest_digest']}, indent=2))


In [ ]:
!cd artifacts/lora-train && zip -r /content/MEL-QLORA-smoke-500.zip smoke-500 >/dev/null
print('Artefact prêt : /content/MEL-QLORA-smoke-500.zip')

## Publication gratuite vers Hugging Face
Après l'entraînement, cette étape publie automatiquement le bundle LoRA sur le dépôt public `Meliturgos/mel-lora-smoke-500`. Au premier lancement, Hugging Face demandera une connexion dans Colab. Aucun secret Cloudflare n'est nécessaire ici.

In [ ]:
!python -m pip install -U huggingface_hub
from huggingface_hub import HfApi, notebook_login
try:
    me = HfApi().whoami()
    print('Hugging Face connecté :', me.get('name') or me.get('fullname'))
except Exception:
    notebook_login()
!python scripts/publish-lora-hf.py --dir artifacts/lora-train/smoke-500 --repo-id Meliturgos/mel-lora-smoke-500
print('Bundle publié : https://huggingface.co/Meliturgos/mel-lora-smoke-500')
print('Étape suivante : ouvrir le mode complet MEL > LoRA gratuit, puis lancer la promotion candidate.')

## Étape Cloudflare (après entraînement)
Pour obtenir un `finetune_id`, définissez `CLOUDFLARE_ACCOUNT_ID` et `CLOUDFLARE_API_TOKEN` dans l'environnement, puis exécutez la cellule suivante. Cette étape charge uniquement les deux fichiers LoRA attendus et met à jour `artifact-evidence.json`. Le benchmark et l'activation restent séparés.


In [ ]:
import os
if os.getenv('CLOUDFLARE_ACCOUNT_ID') and os.getenv('CLOUDFLARE_API_TOKEN'):
    !node scripts/upload-cloudflare-lora.mjs --dir artifacts/lora-train/smoke-500 --name mel-smoke-500
else:
    print('Upload Cloudflare non lancé : définir CLOUDFLARE_ACCOUNT_ID et CLOUDFLARE_API_TOKEN.')
